# 04a — Analysis plan (pre-registration) for steps 4–7

**Lymphoma AI Research** · Step 4, part a

This notebook writes the analysis plan, shows it for review and then **locks** it. The plan is saved twice: as Markdown (for people) and as JSON (the parameters the analysis code reads). Their SHA-256 hashes go into a lock file, together with a timestamp and the hashes of every input file. The model-fitting notebook (04b) reads its settings from the locked JSON and refuses to run if the file was changed.

This notebook fits no model and computes no survival association. The test-set and external expression files are **hashed, never parsed**: the hash proves later that they were not changed, without looking inside.

TR: Analiz planını yazar, okuman için gösterir ve kilitler. Kilit = planın SHA-256 özeti + zaman damgası + girdi dosyalarının özetleri. Model kurmaz. Test ve dış set dosyalarını açmaz, yalnızca özetlerini alır.

## 1. Setup
TR: Drive'ı bağlar, klasör yollarını ve SHA-256 yardımcı fonksiyonlarını tanımlar. Ek paket kurmaz.

In [ ]:
import os, sys, json, re, hashlib, datetime, platform
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/dlbcl-prognosis'
except ImportError:                      # outside Colab (local test runs)
    PROJECT_DIR = os.environ.get('DLBCL_PROJECT_DIR', os.path.abspath('dlbcl-prognosis'))

assert os.path.isdir(PROJECT_DIR), f'Project folder not found: {PROJECT_DIR}'
CLEAN = f'{PROJECT_DIR}/data/clean'
PROC  = f'{PROJECT_DIR}/data/processed'
RES   = f'{PROJECT_DIR}/results/04'
os.makedirs(RES, exist_ok=True)

PLAN_JSON_PATH = f'{RES}/analysis_plan_04.json'
PLAN_MD_PATH   = f'{RES}/analysis_plan_04.md'
LOCK_PATH      = f'{RES}/plan_lock_04.json'
AMEND_PATH     = f'{RES}/plan_amendments.md'
INVENTORY_PATH = f'{RES}/inventory_04a.json'

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def sha256_text(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

print('Project folder:', PROJECT_DIR)
print('Plan already locked:', os.path.exists(LOCK_PATH))

## 2. Inputs: check and fingerprint
Every input of steps 4–7 must exist. Each file is hashed (SHA-256). The hashes are compared with the ones recorded by steps 2a, 2b and 3, and they go into the lock file. The test-set and external expression files are only hashed here; their contents are not read.

TR: Önceki adımların çıktıları yerinde mi diye bakar ve her dosyanın SHA-256 özetini alır. Özetler 2a/2b/3 manifestolarındaki kayıtlarla karşılaştırılır. "match" = dosya o adımdan beri değişmemiş. "not listed" = o manifesto bu dosyayı kaydetmemiş; tek başına sorun değil.

In [ ]:
INPUTS = {   # key: (path inside PROJECT_DIR, how steps 4-7 use it)
    'clinical_all':  ('data/clean/clinical_clean_all.csv',           'step 4: training rows only'),
    'split':         ('data/clean/split_GSE181063.csv',              'read'),
    'manifest_02a':  ('data/clean/manifest_02a.json',                'read'),
    'expr_train':    ('data/processed/expr_GSE181063_train.parquet', 'step 4'),
    'gene_table':    ('data/processed/gene_table_common.csv',        'read'),
    'qc_samples':    ('data/processed/qc_samples.csv',               'step 4: training rows only'),
    'manifest_02b':  ('data/processed/manifest_02b.json',            'read'),
    'plan_03':       ('results/03/plan_03.json',                     'read'),
    'manifest_03':   ('results/03/manifest_03.json',                 'read'),
    'expr_test':     ('data/processed/expr_GSE181063_test.parquet',  'hash only; opened in step 5'),
    'expr_GSE10846': ('data/processed/expr_GSE10846.parquet',        'hash only; opened in step 6'),
    'expr_GSE87371': ('data/processed/expr_GSE87371.parquet',        'hash only; opened in step 6'),
}
missing = [rel for rel, _ in INPUTS.values() if not os.path.exists(f'{PROJECT_DIR}/{rel}')]
if missing:
    raise FileNotFoundError('Missing inputs (check the path or file name):\n  ' + '\n  '.join(missing))

input_sha = {k: sha256_file(f'{PROJECT_DIR}/{rel}') for k, (rel, _) in INPUTS.items()}

def hex_strings(obj):
    # every hash-like string recorded anywhere inside a manifest
    if isinstance(obj, dict):
        for v in obj.values():
            yield from hex_strings(v)
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            yield from hex_strings(v)
    elif isinstance(obj, str) and re.fullmatch(r'[0-9a-f]{12,64}', obj):
        yield obj

recorded = set()
for k in ('manifest_02a', 'manifest_02b', 'manifest_03'):
    with open(f'{PROJECT_DIR}/{INPUTS[k][0]}') as f:
        recorded |= set(hex_strings(json.load(f)))

rows = []
for k, (rel, use) in INPUTS.items():
    if k.startswith('manifest'):
        status = '-'
    else:
        status = 'match' if any(input_sha[k].startswith(r) for r in recorded) else 'not listed'
    rows.append({'input': k, 'use': use, 'sha256 (first 12)': input_sha[k][:12], 'earlier manifest': status})
INPUT_CHECK = {r['input']: r['earlier manifest'] for r in rows if r['earlier manifest'] != '-'}
display(pd.DataFrame(rows))

## 3. The plan: parameters (machine-readable)
All settings live in one dictionary. 04b reads them from the saved JSON, so the analysis cannot silently drift from the plan. Patient and death counts are before the QC rule.

TR: Planın bütün ayarları tek bir sözlükte. 04b bu ayarları kilitli JSON dosyasından okuyacak; dosya değişirse 04b çalışmaz. Bu hücreyi kilitten sonra değiştirme.

In [ ]:
SEED = 20261015

PLAN = {
    'plan': 'dlbcl-prognosis analysis plan for steps 4-7',
    'version': '1.0',
    'drafted_on': '2026-10-15',
    'master_seed': SEED,

    'cohorts': {   # counts before the QC rule
        'training':            {'gse': 'GSE181063', 'split': 'train', 'platform': 'GPL14951',
                                'patients': 380, 'deaths': 173, 'first_used_in_step': 4},
        'internal_test':       {'gse': 'GSE181063', 'split': 'test', 'platform': 'GPL14951',
                                'patients': 164, 'deaths': 74, 'first_used_in_step': 5},
        'external_main':       {'gse': 'GSE10846', 'platform': 'GPL570',
                                'patients': 232, 'deaths': 59, 'first_used_in_step': 6},
        'external_supportive': {'gse': 'GSE87371', 'platform': 'GPL570',
                                'patients': 116, 'deaths': 41, 'first_used_in_step': 6},
    },
    'qc': {
        'primary_exclusion_flag': 'sex_mismatch_confident',
        'sensitivity_exclusion_flag': 'corr_outlier',
        'expected_exclusions': {'training': 7, 'internal_test': 2, 'GSE10846': 3, 'GSE87371': 0},
        'applies_to_all_cohorts': True,
    },
    'endpoints': {
        'primary': {'name': 'OS', 'time': 'os_time_months', 'event': 'os_event',
                    'event_definition': 'death from any cause', 'administrative_truncation': None},
        'secondary': {'name': 'PFS',
                      'cohorts': 'only where the raw GEO records provide PFS; extracted with the step 2a inclusion rules; coding checked before any PFS analysis',
                      'model_trained_on_pfs': False},
        'uno_c_tau_months': 60,
        'uno_c_min_at_risk_fraction': 0.20,
    },
    'features': {
        'gene_table': 'data/processed/gene_table_common.csv',
        'gene_id': 'entrez_id',
        'n_genes_input': 13747,
        'variance_filter_top_n': 5000,
        'variance_ddof': 1,
        'standardization': 'gene-wise z-score with the mean and SD (ddof=1) of the training data of the current fit',
        'refit_inside_every_cv_fold': True,
        'supervised_prescreening': False,
        'clinical_variables_as_model_inputs': False,
    },
    'model': {
        'type': 'Cox proportional hazards with elastic-net penalty',
        'implementation': 'scikit-survival CoxnetSurvivalAnalysis',
        'l1_ratio': 0.9,
        'n_alphas': 100,
        'alpha_min_ratio': 0.01,
        'ties': 'breslow',
        'cv_folds': 10,
        'cv_stratified_by': 'os_event',
        'cv_criterion': 'cross-validated partial-likelihood deviance (Verweij and van Houwelingen 1993; grouped, as in glmnet)',
        'lambda_rule_primary': 'lambda_min',
        'lambda_rule_sensitivity': 'lambda_1se',
        'numerical_failure_rule': 'penalty values at which any fold fails are dropped for all folds; reported',
        'empty_model_rule': 'reported as a null result; settings are not changed to force genes into the model',
        'grid_boundary_rule': 'if lambda_min is the smallest penalty on the grid, this is reported; the grid is not extended',
        'risk_score': 'linear predictor LP = sum over selected genes of beta_g * z_g',
    },
    'risk_groups': {
        'primary_cutoff': 'median LP of the training set; high vs low',
        'internal_test_cutoff': 'the numeric training cutoff',
        'external_cutoff': 'median LP of each external cohort',
        'descriptive': 'tertiles',
        'optimal_cutpoint_search': False,
    },
    'step4_training': {
        'nested_cv': {'outer_folds': 5, 'outer_repeats': 5, 'stratified_by': 'os_event',
                      'metric': "Harrell's C in each outer test fold",
                      'lambda_rules': ['lambda_min', 'lambda_1se'],
                      'cv_km': 'repeat 1; each outer test fold split at the median LP of its outer training fold'},
        'added_value_descriptive': ('out-of-fold LP (repeat 1, standardized with the LP mean and SD of its outer '
                                    'training fold) added to Cox(IPI + COO) in IPI complete cases; LRT and delta C'),
        'stability_bootstrap': {'resamples': 100, 'penalty': 'fixed at lambda_min of the final model'},
        'sensitivity': {
            'a_lambda_1se': 'same CV as the primary model',
            'b_pure_lasso': {'l1_ratio': 1.0, 'nested_cv': '5-fold x 1 repeat, same folds as primary repeat 1'},
            'c_no_variance_filter': {'variance_filter_top_n': None,
                                     'nested_cv': '5-fold x 1 repeat, same folds as primary repeat 1'},
            'd_exclude_corr_outliers': 'final model only',
            'e_alternative_cv_seeds': 'final model only, 5 seeds',
            'reported': ['number of genes', 'Jaccard overlap with the primary model', 'nested-CV C where computed'],
            'taken_to_test_set': False,
        },
        'outputs': ['results/04/model_04.json (frozen model)', 'CV deviance curve', 'nested-CV results',
                    'gene stability table', 'training sensitivity table', 'results/04/manifest_04b.json'],
    },
    'step5_internal_validation': {
        'precondition': 'SHA-256 of the frozen model recorded in manifest_04b.json',
        'test_scaling': 'training means and SDs',
        'test_scaling_sensitivity': 'z-score within the test set (primary population)',
        'primary_metric': "Cox HR per 1 SD of LP (SD of the cohort's own LP) with 95% CI",
        'other_metrics': ["Harrell's C with 95% bootstrap CI (percentile)",
                          "Uno's C truncated at tau (endpoints: uno_c_tau_months, uno_c_min_at_risk_fraction)",
                          'Kaplan-Meier high vs low risk (training cutoff), log-rank test, HR high vs low',
                          'calibration slope (Cox coefficient of the raw LP)',
                          'proportional-hazards check (Schoenfeld residuals), descriptive'],
        'bootstrap_resamples': 1000,
        'H1': 'HR per SD > 1 with 95% CI excluding 1 (two-sided alpha = 0.05)',
        'supporting': ['C-index 95% CI excludes 0.5', 'log-rank p < 0.05', 'permutation p < 0.05'],
        'qc_sensitivity': 'correlation outliers excluded (no re-scaling)',
    },
    'step5_negative_control': {
        'permutation': '(time, event) pairs permuted jointly among the primary training patients',
        'n_permutations': 100,
        'procedure': 'full pipeline of plan sections 4-5, including CV for the penalty, re-run on each permutation',
        'evaluation': 'C-index on the true test labels',
        'report': ['mean and 2.5-97.5 percentiles of permuted test C', 'share of empty models',
                   'empirical p = (1 + number of permuted C >= observed C) / (1 + n_permutations)'],
        'leakage_check': {'n_permutations': 10, 'nested_cv': '5-fold x 1 repeat', 'stop_if_mean_c_above': 0.55},
    },
    'step6_external_validation': {
        'transfer': 'frozen Entrez IDs and coefficients; no re-selection, re-fitting or re-tuning',
        'scaling': 'gene-wise z-score within each cohort, computed on its primary analysis population',
        'not_transferred': 'absolute scale: baseline hazard and numeric LP cutoff',
        'risk_group_cutoff': 'median LP of each cohort',
        'metrics': 'as step 5, except calibration slope',
        'H2': 'per cohort: HR per SD > 1 with 95% CI excluding 1; GSE10846 main, GSE87371 supportive',
        'sensitivity': ['GSE87371 without R-mini-CHOP21 patients (n = 13)', 'correlation outliers excluded'],
        'sensitivity_scaling': 'scores of the primary analysis; no re-scaling',
    },
    'step7_clinical_comparison': {
        'benchmarks': ['IPI (0-5, linear, 1 df)', 'COO (GCB reference; ABC; other = UNC, MHG or unclassified)',
                       'IPI + COO'],
        'H3': ('likelihood-ratio test (1 df), Cox(IPI + COO) vs Cox(IPI + COO + LP per SD); '
               'primary in the test set, p < 0.05'),
        'also_reported': ['adjusted HR per SD with 95% CI',
                          "delta Harrell's C with 95% bootstrap CI (both models refitted in each resample)"],
        'delta_c_note': ('combination weights are fitted in the same cohort, which slightly favours the larger model; '
                         'the likelihood-ratio test accounts for the extra parameter'),
        'bootstrap_resamples': 1000,
        'missing_ipi': 'complete-case analysis with missingness diagnostics',
        'coo_sensitivity': 'GSE181063: MHG as a separate category',
        'descriptive': 'C-index of IPI, COO and LP alone and combined, in the same complete-case subset',
        'training_reference_from_step3': {'subset': 'ABC/GCB patients with known IPI', 'n': 223,
                                          'coo_alone_c': 0.614, 'ipi_plus_coo_c': 0.736},
    },
    'conclusion_rules': {
        'internally_validated': 'H1 met',
        'externally_validated': 'H2 met in GSE10846 and HR > 1 in GSE87371',
        'adds_information_beyond_ipi_and_coo': 'H3 met in the test set and adjusted HR > 1 in both external cohorts',
        'otherwise': 'reported as negative or inconclusive, with the same detail',
    },
    'red_flags': ['any C-index > 0.90', 'permuted-label mean C > 0.55',
                  'test-set C exceeds the nested-CV mean C by more than 0.10'],
    'power': {   # Schoenfeld: power = Phi(sqrt(deaths) * ln(HR) - 1.96); score per SD; two-sided alpha 0.05
        'GSE181063 test': {'deaths': 74, 'min_hr_per_sd_80pct': 1.38, 'power_hr_1_3': 0.62, 'power_hr_1_5': 0.94},
        'GSE10846':       {'deaths': 59, 'min_hr_per_sd_80pct': 1.44, 'power_hr_1_3': 0.52, 'power_hr_1_5': 0.88},
        'GSE87371':       {'deaths': 41, 'min_hr_per_sd_80pct': 1.55, 'power_hr_1_3': 0.39, 'power_hr_1_5': 0.74},
    },
    'multiplicity': 'H1 is the only primary hypothesis; all other tests are secondary, unadjusted and supportive',
    'seeds': {
        'final_model_cv_folds': SEED,
        'alternative_cv_seeds': [SEED + i for i in range(1, 6)],
        'nested_cv_outer_split': 'SEED + 100 + r (repeat r = 0..4)',
        'nested_cv_inner_folds': 'SEED + 200 + 10*r + j (outer fold j = 0..4)',
        'stability_bootstrap': SEED + 300,
        'validation_bootstrap': SEED + 400,
        'delta_c_bootstrap': SEED + 500,
        'permutation_labels': 'SEED + 1000 + i (i = 0..99)',
        'permutation_cv_folds': 'SEED + 2000 + i',
        'leakage_check_labels': 'SEED + 3000 + i (i = 0..9)',
        'leakage_check_outer_split': 'SEED + 3100 + i',
        'leakage_check_inner_folds': 'SEED + 3200 + 10*i + j',
    },
    'data_access': {
        'step4_uses': ['training rows of clinical_clean_all.csv (other rows dropped right after loading)', 'split_GSE181063.csv',
                        'expr_GSE181063_train.parquet', 'training rows of qc_samples.csv', 'gene_table_common.csv'],
        'step4_never_opens': ['expr_GSE181063_test.parquet', 'expr_GSE10846.parquet', 'expr_GSE87371.parquet'],
        'test_set_opened_in': 'step 5, after the frozen model hash is recorded',
        'external_cohorts_opened_in': 'step 6',
    },
    'deviations': {'file': 'results/04/plan_amendments.md',
                   'policy': ('locked files are never edited; changes are appended with date, reason and data seen; '
                              'after the test set is opened only error corrections are allowed and both the '
                              'original and the corrected results are reported')},
    'reporting': ['TRIPOD+AI', 'REMARK'],
}
print('Plan parameters defined:', len(PLAN), 'sections')

## 4. The plan: document
The readable version of the same plan. Every number is taken from the dictionary above, so the two versions cannot disagree. **Read it in full before locking.**

TR: Aynı planın okunabilir metni. Sayılar yukarıdaki sözlükten geliyor; iki sürüm birbirini tutmak zorunda. Kilitlemeden önce baştan sona oku.

In [ ]:
P = PLAN
CO, Q, F, M = P['cohorts'], P['qc'], P['features'], P['model']
S4, S5, NC = P['step4_training'], P['step5_internal_validation'], P['step5_negative_control']
S7 = P['step7_clinical_comparison']
NCV, EX, TR3 = S4['nested_cv'], Q['expected_exclusions'], S7['training_reference_from_step3']
TAU, UNO_MIN = P['endpoints']['uno_c_tau_months'], P['endpoints']['uno_c_min_at_risk_fraction']
NPERM, NLEAK = NC['n_permutations'], NC['leakage_check']['n_permutations']
LEAK_MAX = NC['leakage_check']['stop_if_mean_c_above']

def n(x):
    return f'{x:,}'

def cohort_row(role, key, label, platform_note, excl):
    c = CO[key]
    return (f"| {role} | {label} | {c['platform']} ({platform_note}) | {c['patients']} | {c['deaths']} "
            f"| {excl} | Step {c['first_used_in_step']} |")

cohort_rows = '\n'.join([
    cohort_row('Training', 'training', 'GSE181063, train split', 'DASL, FFPE', EX['training']),
    cohort_row('Internal test', 'internal_test', 'GSE181063, test split', 'DASL, FFPE', EX['internal_test']),
    cohort_row('External, main', 'external_main', 'GSE10846', 'HG-U133 Plus 2.0', EX['GSE10846']),
    cohort_row('External, supportive', 'external_supportive', 'GSE87371', 'HG-U133 Plus 2.0', EX['GSE87371']),
])
power_rows = '\n'.join(
    f"| {k} | {v['deaths']} | {v['min_hr_per_sd_80pct']:.2f} | {v['power_hr_1_3']:.2f} | {v['power_hr_1_5']:.2f} |"
    for k, v in P['power'].items())

PLAN_MD = f"""# Analysis plan — steps 4–7 (version {P['version']})

**Project:** Lymphoma AI Research (`dlbcl-prognosis`) · **Drafted:** {P['drafted_on']} · **Master seed:** {P['master_seed']}

**Status:** locked when `results/04/plan_lock_04.json` exists. The lock file stores the SHA-256 of this document and of `analysis_plan_04.json` (the parameters the analysis code reads), a timestamp, and the hashes of all input files.

This is a research project. No part of it is a diagnostic or treatment tool, and no result will be presented as one.

## 0. Data seen before this plan

| Data | Seen so far | Outcome data used so far |
|---|---|---|
| GSE181063 training (n = {CO['training']['patients']}) | clinical cleaning, expression QC | descriptive counts (Step 2a); OS by cell of origin, with and without IPI adjustment (Step 3 positive control) |
| GSE181063 test (n = {CO['internal_test']['patients']}) | clinical cleaning, expression QC | descriptive counts only (Step 2a) |
| GSE10846, GSE87371 | clinical cleaning, expression QC | descriptive counts only (Step 2a) |

Descriptive counts: number of deaths and median follow-up per cohort. No association between any gene and survival has been computed in any cohort, and no outcome comparison has been made outside the training set.

## 1. Objectives

- **Primary.** Develop a penalized Cox model of overall survival (OS) from gene expression in the GSE181063 training set, freeze it, and test it once in the held-out GSE181063 test set (Step 5).
- **Secondary.** Test the frozen model in two external cohorts measured on a different platform (Step 6), and test whether it adds prognostic information to the IPI and cell of origin (COO) (Step 7).
- **Exploratory.** Biological interpretation of the selected genes (Step 8). Its methods will be set in Step 8; it is not confirmatory.

## 2. Cohorts and analysis populations

| Role | Cohort | Platform | Patients | Deaths | Expected QC exclusions | First used |
|---|---|---|---|---|---|---|
{cohort_rows}

Patient and death counts are before the QC rule. Inclusion criteria are those fixed in Step 2a.

- **QC rule (primary, all cohorts):** exclude samples flagged `{Q['primary_exclusion_flag']}` in Step 2b. Their clinical data may belong to another sample.
- **QC sensitivity analysis:** additionally exclude samples flagged `{Q['sensitivity_exclusion_flag']}`.
- All COO groups, including MHG and UNC, are used for model development. COO is not a model input.

## 3. Endpoints

- **Primary:** OS in months; event = death from any cause; censored at last follow-up; time origin as defined by the data providers. No administrative truncation during model development.
- **Secondary:** progression-free survival (PFS), only in cohorts whose GEO records provide it. PFS is not in the Step 2a clean files; if used, it is extracted from the raw GEO tables with the same inclusion rules, and its coding is checked (as in Step 2a) before any PFS analysis. The frozen OS model is applied unchanged. No model is trained on PFS.

## 4. Features and pre-processing

1. Input: the {n(F['n_genes_input'])} genes common to GPL14951 and GPL570 (Step 2b), log2 and quantile-normalized, identified by **Entrez Gene ID**. Gene symbols are for display only.
2. Variance filter: keep the {n(F['variance_filter_top_n'])} genes with the highest variance in the training data of the current fit.
3. Standardization: gene-wise z-score with the mean and SD of the training data of the current fit.
4. "Current fit" means each cross-validation training fold, or the full training set for the final model. Steps 2–3 are re-estimated inside every fold, so held-out samples never influence them.
5. No supervised pre-screening (for example, univariable Cox filtering). Survival data are used only inside the penalized Cox fit.
6. Clinical variables (IPI, COO, age, sex) are not model inputs. They are used only for comparison (Step 7).

## 5. Model

- Cox proportional-hazards model with an elastic-net penalty, `l1_ratio` = {M['l1_ratio']}: mostly LASSO; the small ridge part makes selection among correlated genes more stable. Fitted with scikit-survival `CoxnetSurvivalAnalysis`: {M['n_alphas']} penalty values λ (called `alpha` in scikit-survival), `alpha_min_ratio` = {M['alpha_min_ratio']}, Breslow ties.
- **Penalty selection:** {M['cv_folds']}-fold cross-validation, folds stratified by OS event, seed {P['seeds']['final_model_cv_folds']}. Criterion: cross-validated partial-likelihood deviance (Verweij & van Houwelingen 1993; the default in glmnet). **λ_min** (lowest mean CV deviance) is primary. λ_1se (largest penalty within one standard error of the minimum) is recorded for sensitivity only.
- **Final model:** fitted on the full QC-filtered training set at λ_min, then frozen and hashed before Step 5: Entrez IDs, coefficients, training means and SDs of the selected genes, penalty value and risk-group cutoff.
- **Risk score:** linear predictor, LP = Σ β_g · z_g over the selected genes.
- **If no gene is selected at λ_min,** this is reported as a null result. Settings are not changed to force genes into the model.
- If λ_min is the smallest penalty on the grid, this is reported and the grid is not extended.
- If the fit fails numerically at some penalty values in any fold, those values are dropped for all folds and this is reported.

## 6. Risk groups

- **Primary:** high vs low risk, split at the median LP of the training set. The same numeric cutoff is used in the GSE181063 test set.
- **External cohorts:** each cohort's own median LP (§10).
- **Descriptive only:** tertiles.
- Data-driven "optimal" cut-points (for example, maximally selected log-rank statistics) are not used.

## 7. Step 4 outputs (training set only)

1. The frozen final model, the CV deviance curve and the coefficients.
2. **Nested cross-validation** (an honest estimate inside the training set): outer {NCV['outer_folds']}-fold × {NCV['outer_repeats']} repeats, stratified by event. The whole procedure of §4–5 runs inside each outer training fold. Reported: mean and SD of Harrell's C over the {NCV['outer_folds'] * NCV['outer_repeats']} outer test folds, for λ_min and λ_1se, and a cross-validated Kaplan–Meier curve (Simon et al. 2011; repeat 1), in which each outer test fold is split at the median LP of its own outer training fold and the folds are then pooled. This estimate is descriptive. It does not decide whether the model goes to the test set: the primary model goes regardless.
3. **Added value inside the training set (descriptive):** the out-of-fold LP (repeat 1, standardized with the LP mean and SD of its outer training fold) added to Cox(IPI + COO) in patients with known IPI; likelihood-ratio test and ΔC.
4. **Gene stability:** {S4['stability_bootstrap']['resamples']} bootstrap resamples; §4 pre-processing and the Cox fit at the fixed λ_min of the final model; selection frequency of each gene. Used for interpretation in Step 8, not for selection.
5. **Training-only sensitivity analyses** (never taken to the test set): (a) λ_1se; (b) pure LASSO (`l1_ratio` = 1.0); (c) no variance filter (all {n(F['n_genes_input'])} genes); (d) correlation outliers excluded; (e) five other CV seeds. Reported for each: number of genes and overlap with the primary model (Jaccard index); for (b) and (c) also nested-CV C (5-fold × 1 repeat, same folds as repeat 1 of the primary analysis).

## 8. Internal validation (Step 5)

Step 5 starts only after the SHA-256 of the frozen model is recorded. The model is not changed afterwards.

1. Test-set expression is z-scored with the **training** means and SDs (same platform, same normalization reference). Sensitivity: z-scoring within the test set.
2. Metrics:
   - **Primary:** Cox hazard ratio (HR) per 1 SD of LP (SD of the cohort's own LP), with 95% CI.
   - Harrell's C with 95% bootstrap CI ({n(S5['bootstrap_resamples'])} resamples, percentile method).
   - Uno's C, truncated at τ = {TAU} months. If fewer than {UNO_MIN:.0%} of a cohort's patients are still at risk (alive and under follow-up) at {TAU} months, τ is the latest time at which at least {UNO_MIN:.0%} are. The censoring distribution is estimated in the same cohort.
   - Kaplan–Meier curves for high vs low risk (training cutoff), log-rank test, HR high vs low.
   - Calibration slope: the Cox coefficient of the raw LP (1 = ideal; below 1 = overfitting).
   - Proportional-hazards check (Schoenfeld residuals), descriptive.
3. **H1 (the primary hypothesis):** HR per SD > 1 with a 95% CI that excludes 1 (two-sided α = 0.05). Supporting evidence: C-index CI excludes 0.5; log-rank p < 0.05; permutation p < 0.05 (§9).
4. QC sensitivity: correlation outliers excluded, without re-scaling.

## 9. Negative control (Step 5)

1. **Label permutation:** survival time and event are permuted together across training patients. This breaks any link with expression but keeps the survival distribution. The full procedure (§4–5, including CV for the penalty) is re-run on each of **{NPERM}** permutations, and each permuted model is scored on the **true** test labels. Reported: mean and 2.5–97.5th percentiles of the permuted test C, the share of empty models, and the empirical p-value = (1 + number of permuted C ≥ observed C) / (1 + {NPERM}). An empty model has C = 0.5 by definition (all scores tied). Expected: permuted C centred at 0.5; observed model p < 0.05.
2. **Leakage check:** nested CV (5-fold × 1 repeat) on {NLEAK} permuted training sets. Expected mean C ≈ 0.5. If the mean is above {LEAK_MAX}, the analysis stops and the pipeline is searched for leakage.

## 10. External validation (Step 6)

1. The **frozen genes and frozen coefficients** are applied unchanged. No re-selection, no re-fitting, no re-tuning.
2. Expression is z-scored **within each cohort**, gene by gene, on its primary analysis population, because absolute expression levels are not comparable between platforms. What is not transferred is the absolute scale: the baseline hazard and the numeric LP cutoff.
3. Risk groups: each cohort's own median LP (the same quantile as in training).
4. Metrics as §8.2, except the calibration slope, which cannot be interpreted after within-cohort scaling.
5. **H2 (per cohort):** HR per SD > 1 with a 95% CI that excludes 1. GSE10846 is the main external test; GSE87371 is supportive (§14).
6. Sensitivity analyses, both using the scores of the primary analysis (no re-scaling): GSE87371 without R-mini-CHOP21 patients (n = 13); correlation outliers excluded in each cohort.

## 11. Comparison with IPI and cell of origin (Step 7)

1. Benchmarks, fitted in each validation cohort: IPI (score 0–5, linear, 1 df); COO (GCB as reference; ABC; other = UNC, MHG or unclassified); IPI + COO.
2. **H3 (added value):** Cox(IPI + COO) vs Cox(IPI + COO + LP per SD), likelihood-ratio test with 1 df. Primary in the test set: p < 0.05. Also reported: adjusted HR per SD with 95% CI; ΔHarrell's C with 95% bootstrap CI ({n(S7['bootstrap_resamples'])} resamples, both models refitted in each resample). Because the combination weights are fitted in the same cohort, ΔC slightly favours the larger model; the likelihood-ratio test accounts for the extra parameter.
3. **Missing IPI:** complete-case analysis. Diagnostics: patients with and without IPI are compared on age, sex, COO, follow-up and deaths, and the univariable HR of LP is reported in both groups.
4. COO sensitivity in GSE181063: MHG as a separate category.
5. Descriptive: C-index of IPI alone, COO alone, LP alone and their combinations, all in the same complete-case subset.
6. Reference point from Step 3 (training set, {TR3['subset']}, n = {TR3['n']}): COO alone C = {TR3['coo_alone_c']}; IPI + COO C = {TR3['ipi_plus_coo_c']}. A gene score is not expected to beat IPI + COO on its own; the question is whether it adds to it (H3).

## 12. Conclusion rules (wording fixed in advance)

- "**Internally validated**": H1 met.
- "**Externally validated**": H2 met in GSE10846, and HR > 1 in GSE87371 (its CI may include 1).
- "**Adds information beyond IPI and COO**": H3 met in the test set, and adjusted HR > 1 in both external cohorts.
- Results that do not meet a rule are reported as negative or inconclusive, with the same detail as positive ones.

## 13. Red flags: stop and look for errors before reporting

- Any C-index > 0.90.
- Mean C of the permuted-label models > {LEAK_MAX} (§9.2).
- Test-set C higher than the nested-CV mean C by more than 0.10.

## 14. Power (approximate)

Schoenfeld approximation for a continuous score per SD, two-sided α = 0.05: power = Φ(√d · ln HR − 1.96), where d = number of deaths.

| Cohort | Deaths | Smallest HR per SD detectable with 80% power | Power at HR 1.3 | Power at HR 1.5 |
|---|---|---|---|---|
{power_rows}

GSE87371 has little power for moderate effects; a non-significant result there is not evidence against the model.

## 15. Multiplicity

H1 is the only primary hypothesis. All other tests are secondary, reported without adjustment and interpreted as supportive.

## 16. Data-access discipline

- Step 4 uses only: the training rows of the clinical table and of the QC table (other rows are dropped right after loading), the split file, the training expression file and the gene table.
- Step 4 never opens the test expression file or the external cohorts. They are only hashed, to prove later that they were not changed.
- The test set is opened in Step 5, after the frozen model's hash is recorded. The external cohorts are opened in Step 6.

## 17. Deviations

- The locked files are never edited. Any change is appended to `{P['deviations']['file']}` with the date, the reason and which data had been seen at that time.
- After the test set is opened, changes are allowed only to correct errors, and both the original and the corrected results are then reported.

## 18. Reproducibility and reporting

- All random seeds derive from the master seed {P['master_seed']}; the scheme is in `analysis_plan_04.json`. Software versions and SHA-256 hashes of inputs and outputs are recorded in each step's manifest.
- The report will follow the TRIPOD+AI and REMARK reporting guidelines.

## 19. Known limitations at the planning stage

- Three unsupervised steps of Step 2b used all 380 training samples, outside cross-validation: the quantile-normalization reference, the choice of one probe per gene (highest training mean) and the low-expression filter. None of them uses outcome data. They can make the nested-CV estimate (§7) very slightly optimistic; they do not bias the test-set results, because their parameters came from training samples only.
- The training data are FFPE tissue on a DASL array; the external cohorts are Affymetrix arrays. Lower external performance is expected.
- Within-cohort z-scoring depends on each cohort's case mix (for example, its share of ABC cases).
- COO classifiers differ between cohorts; MHG exists only in GSE181063.
- IPI is missing for part of GSE181063 and GSE10846; complete-case analyses can be biased if missingness is not random.
- The sex-check rule (Step 2b) was set after seeing the training QC results, without outcome data.
- The sensitivity analysis that includes non-pre-treatment biopsies (flag set in Step 2a; the exact meaning of that GEO field is uncertain) is not part of this plan, because it would require re-running Steps 2a–2b.

## References

- Collins GS, et al. TRIPOD+AI statement: updated guidance for reporting clinical prediction models that use regression or machine learning methods. BMJ 2024;385:e078378.
- Harrell FE, Lee KL, Mark DB. Multivariable prognostic models: issues in developing models, evaluating assumptions and adequacy, and measuring and reducing errors. Stat Med 1996;15:361–387.
- McShane LM, et al. Reporting recommendations for tumor marker prognostic studies (REMARK). J Natl Cancer Inst 2005;97:1180–1184.
- Pölsterl S. scikit-survival: a library for time-to-event analysis built on top of scikit-learn. J Mach Learn Res 2020;21(212):1–6.
- Schoenfeld DA. Sample-size formula for the proportional-hazards regression model. Biometrics 1983;39:499–503.
- Simon N, Friedman J, Hastie T, Tibshirani R. Regularization paths for Cox's proportional hazards model via coordinate descent. J Stat Softw 2011;39(5):1–13.
- Simon RM, Subramanian J, Li MC, Menezes S. Using cross-validation to evaluate predictive accuracy of survival risk classifiers based on high-dimensional data. Brief Bioinform 2011;12:203–214.
- Uno H, Cai T, Pencina MJ, D'Agostino RB, Wei LJ. On the C-statistics for evaluating overall adequacy of risk prediction procedures with censored survival data. Stat Med 2011;30:1105–1117.
- Verweij PJM, van Houwelingen HC. Cross-validation in survival analysis. Stat Med 1993;12:2305–2314.
"""
display(Markdown(PLAN_MD))

## 5. Lock the plan
On the first run this cell only prints **NOT LOCKED YET**. That is expected, not an error. Read the plan in section 4. If you approve it, change the first line of the cell below to `REVIEWED_AND_APPROVED = True`, run the cell again, then run section 6. The cell writes the plan files first and the lock file last, and only if no lock exists yet. If the plan is already locked, it checks that nothing has changed and writes nothing; if something has changed, it stops with an error.

TR: İlk çalıştırmada bu hücre sadece "NOT LOCKED YET" yazar; bu bir hata değil. Planı okuyup onaylıyorsan hücrenin ilk satırını `REVIEWED_AND_APPROVED = True` yap, hücreyi tekrar çalıştır, ardından 6. bölümü çalıştır. Önce plan dosyaları, en son kilit dosyası yazılır. Kilit zaten varsa hiçbir şey yazmaz, yalnızca planın değişmediğini kontrol eder; değiştiyse hata verip durur.

In [ ]:
REVIEWED_AND_APPROVED = True   # <- set to True after reading the plan in section 4

plan_json_text = json.dumps(PLAN, indent=2, ensure_ascii=False) + '\n'
plan_md_text = PLAN_MD
h_json, h_md = sha256_text(plan_json_text), sha256_text(plan_md_text)

def write_text(path, text):
    with open(path, 'w', encoding='utf-8', newline='\n') as f:
        f.write(text)

if os.path.exists(LOCK_PATH):
    with open(LOCK_PATH) as f:
        lock = json.load(f)
    unchanged = (lock['plan_json_sha256'] == h_json and lock['plan_md_sha256'] == h_md
                 and sha256_file(PLAN_JSON_PATH) == h_json and sha256_file(PLAN_MD_PATH) == h_md)
    if not unchanged:
        raise RuntimeError('This plan differs from the locked plan, or a locked file was edited. '
                           'A locked plan is never changed: record changes in plan_amendments.md instead.')
    print(f"Plan already locked at {lock['locked_at_local']}. Unchanged; nothing written.")
elif not REVIEWED_AND_APPROVED:
    print('NOT LOCKED YET. This is expected; nothing is wrong.\n'
          '  1. Read the plan in section 4.\n'
          '  2. If you approve it, change the first line of this cell to: REVIEWED_AND_APPROVED = True\n'
          '  3. Run this cell again, then the cell in section 6.')
else:
    write_text(PLAN_JSON_PATH, plan_json_text)
    write_text(PLAN_MD_PATH, plan_md_text)
    assert sha256_file(PLAN_JSON_PATH) == h_json and sha256_file(PLAN_MD_PATH) == h_md, 'write check failed'
    if not os.path.exists(AMEND_PATH):
        write_text(AMEND_PATH,
                   '# Amendments to analysis_plan_04 (append-only)\n\n'
                   f'Locked plan: analysis_plan_04.md (SHA-256 {h_md[:12]}), analysis_plan_04.json (SHA-256 {h_json[:12]}).\n'
                   'Never edit the locked files. Add one entry per change:\n\n'
                   '## Amendment N (YYYY-MM-DD)\n- Section(s):\n- Change:\n- Reason:\n'
                   '- Data seen when the change was made (training / test / external):\n'
                   '- Effect on results already produced:\n')
    now_utc = datetime.datetime.now(datetime.timezone.utc)
    try:
        from zoneinfo import ZoneInfo
        now_local = now_utc.astimezone(ZoneInfo('Europe/Istanbul'))
    except Exception:
        now_local = now_utc
    lock = {
        'plan': PLAN['plan'], 'version': PLAN['version'],
        'locked_at_utc': now_utc.isoformat(timespec='seconds'),
        'locked_at_local': now_local.isoformat(timespec='seconds'),
        'plan_json_file': 'results/04/analysis_plan_04.json', 'plan_json_sha256': h_json,
        'plan_md_file': 'results/04/analysis_plan_04.md', 'plan_md_sha256': h_md,
        'input_files': {k: rel for k, (rel, _) in INPUTS.items()},
        'input_sha256': input_sha,
        'statement': ('Locked before any model fitting and before the GSE181063 test set or the external '
                      'cohorts were used in any outcome analysis.'),
        'environment': {'python': sys.version.split()[0], 'platform': platform.platform(),
                        'numpy': np.__version__, 'pandas': pd.__version__},
    }
    with open(LOCK_PATH, 'x') as f:      # 'x' = never overwrite an existing lock
        json.dump(lock, f, indent=2)
    print('PLAN LOCKED at', lock['locked_at_local'])

if os.path.exists(LOCK_PATH):
    print('plan JSON SHA-256:', h_json[:12], '| plan MD SHA-256:', h_md[:12])
    print('lock file SHA-256:', sha256_file(LOCK_PATH)[:12])

## 6. Training-set inventory (runs only after the lock)
Before the lock, this cell only prints "Skipped". After the lock, it counts the primary analysis population under the locked QC rule, using training rows only. Test and external rows are dropped right after loading. The expression file is not loaded; only its column names are read, to check that its samples are exactly the training patients. The printed block also repeats the input check of section 2 and lists the column names that 04b will be written against.

TR: Kilitten önce yalnızca "Skipped" yazar. Kilitten sonra, yalnızca eğitim setinde ve kilitli QC kuralıyla birincil popülasyonu sayar: hasta, ölüm, IPI eksikliği, COO dağılımı. Test ve dış set satırları okunur okunmaz bellekten silinir. İfade dosyasının yalnızca sütun adlarına bakılır. En alttaki bloğu bana yapıştır. 2. bölümdeki dosya kontrolü de bu bloğun içinde, ayrıca ekran görüntüsü gerekmez.

In [ ]:
import pyarrow.parquet as pq

def as_bool(s):
    if s.dtype == bool:
        return s
    return s.astype(str).str.strip().str.lower().isin(['true', '1', '1.0', 'yes'])

def counts(s):
    return {str(k): int(v) for k, v in s.fillna('missing').value_counts().items()}

def build_inventory():
    clin_all = pd.read_csv(f'{CLEAN}/clinical_clean_all.csv')
    split = pd.read_csv(f'{CLEAN}/split_GSE181063.csv')
    clinical_columns = list(clin_all.columns)
    train_ids = set(split.loc[split['set'] == 'train', 'sample_id'])
    tr = clin_all[clin_all['sample_id'].isin(train_ids)].copy()
    del clin_all, split                      # test and external rows leave memory here

    qc_all = pd.read_csv(f'{PROC}/qc_samples.csv')
    qc_columns = list(qc_all.columns)
    qc = qc_all[qc_all['sample_id'].isin(train_ids)].copy()
    del qc_all

    TRN = PLAN['cohorts']['training']
    assert len(tr) == TRN['patients'], f'expected {TRN["patients"]} training patients, found {len(tr)}'
    assert int(tr['os_event'].sum()) == TRN['deaths'], 'training death count differs from step 2a'
    assert tr['os_event'].isin([0, 1]).all() and (tr['os_time_months'] > 0).all()
    FP, FS = PLAN['qc']['primary_exclusion_flag'], PLAN['qc']['sensitivity_exclusion_flag']
    for flag in (FP, FS):
        assert flag in qc.columns, f'QC column {flag!r} not found. Available: {qc_columns}'
    assert qc['sample_id'].is_unique and set(qc['sample_id']) == train_ids, \
        'QC table does not match the training samples'

    tr = tr.merge(qc[['sample_id', FP, FS]], on='sample_id', how='left', validate='one_to_one')
    tr[FP], tr[FS] = as_bool(tr[FP]), as_bool(tr[FS])
    prim = tr[~tr[FP]]

    expr_path = f'{PROC}/expr_GSE181063_train.parquet'
    schema = pq.read_schema(expr_path)
    index_cols = [c if isinstance(c, str) else c.get('name')
                  for c in (schema.pandas_metadata or {}).get('index_columns', [])]
    expr_samples = [c for c in schema.names if c not in index_cols]
    n_genes = pq.ParquetFile(expr_path).metadata.num_rows

    return {
        'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
        'plan_lock_sha256': sha256_file(LOCK_PATH),
        'input_check': globals().get('INPUT_CHECK'),
        'training_all': {'patients': int(len(tr)), 'deaths': int(tr['os_event'].sum())},
        'excluded_by_primary_qc': {'patients': int(tr[FP].sum()), 'deaths': int(tr.loc[tr[FP], 'os_event'].sum()),
                                   'sample_ids': sorted(tr.loc[tr[FP], 'sample_id'].astype(str))},
        'primary_population': {'patients': int(len(prim)), 'deaths': int(prim['os_event'].sum()),
                               'ipi_missing': int(prim['ipi'].isna().sum()), 'coo': counts(prim['coo']),
                               'corr_outliers_retained': int(prim[FS].sum())},
        'expression_train': {'genes': int(n_genes), 'samples': len(expr_samples), 'index_columns': index_cols,
                             'samples_equal_training_ids': set(expr_samples) == train_ids,
                             'n_genes_as_planned': int(n_genes) == PLAN['features']['n_genes_input']},
        'schema': {'clinical_columns': clinical_columns, 'qc_columns': qc_columns},
    }

if not os.path.exists(LOCK_PATH):
    print('Skipped: the plan is not locked yet (section 5).')
else:
    inventory = build_inventory()
    with open(INVENTORY_PATH, 'w') as f:
        json.dump(inventory, f, indent=2)
    ok = inventory['expression_train']
    if not (ok['samples_equal_training_ids'] and ok['n_genes_as_planned']):
        print('WARNING: expression file does not match the plan; check before 04b.')
    print('Saved:', INVENTORY_PATH)
    print('\n----- copy everything below and paste it to Claude -----')
    for k, v in inventory.items():
        if k != 'created_utc':
            print(f'{k}: {json.dumps(v)}')

## 7. Next
1. Paste the block printed above into the chat.
2. Close this Colab session (**Runtime → Disconnect and delete runtime**) to save free-tier time.
3. Optional but recommended: upload `analysis_plan_04.md` and `plan_lock_04.json` to the project's GitHub repository now. A public commit is an independent timestamp that proves the plan existed before the results.

TR: Yukarıdaki bloğu sohbete yapıştır ve Colab oturumunu kapat. İstersen planı ve kilit dosyasını şimdi GitHub'a yükle: herkese açık bir commit, planın sonuçlardan önce yazıldığını bağımsız olarak kanıtlar.